# Member B: kiểm chứng B01–B03
Mục tiêu: tự kiểm tra đơn vị, chronology và công thức scoring bằng các bước nhỏ. Notebook chỉ đọc bằng chứng đã có và tạo fixture nhỏ trong thư mục tạm; không chạy benchmark, mở holdout hoặc train model.

**Tiêu chí trước khi chạy:** chuyển normalized → native phải khớp; timestamp tăng đúng 300 giây; scorer từ chối thiếu scenario; score chỉ công bố khi đủ suite 8.640 bước. `pid_water` là policy provisional, A đang phụ trách Base cuối.

> **Nguồn và AI:** Notebook/code B được phát triển với hỗ trợ Codex/ChatGPT; Member B chịu trách nhiệm đọc hiểu và kiểm chứng. Xem `team/PROJECT_KNOWLEDGE.md` về source URL/SHA/license, phần kế thừa và thời gian.
> Base kỹ thuật: official pid_water; member_a là Base + zone logic A tại SHA 3dd6b3a. B04 Ridge là mô hình nghiên cứu offline; B05 chưa được tích hợp.
> Thời điểm cập nhật tri thức: 26/11/2026. Wall/CPU lấy từ receipts; person-hours chưa có worklog xác nhận. DEV-SMALL chỉ có mean khi đủ 5 case mỗi policy; xem team/DEV_SMALL_RESULTS.json.


In [1]:
from pathlib import Path
import sys, csv, json
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "team_tools/benchmark.py").is_file())
sys.path.insert(0, str(ROOT))
print("Repository:", ROOT.name)


Repository: smart-buildings-hvac-2026


## 1. B01: đọc dictionary và khôi phục Kelvin
`dict` ánh xạ tên cột → giá trị; `list` chứa các dòng. `Path` giúp ghép đường dẫn an toàn trên Windows. CSV liệt kê từng field với measurement key thật, đơn vị native, normalizer, dtype và nguồn code. Dữ liệu raw dùng ở audit offline, không đưa vào feature inference.


In [2]:
with (ROOT / "team/data_dictionary_fields.csv").open(encoding="utf-8", newline="") as stream:
    fields = list(csv.DictReader(stream))
print("Số dòng:", len(fields))
print(dict(Counter((row["scenario"], row["kind"]) for row in fields)))
zone = next(row for row in fields if row["kind"] == "observation" and row["field"] == "VAV_1_zone_air_temperature_sensor")
kelvin = float(zone["observed_at_reset"]) * float(zone["std"]) + float(zone["mean"])
assert abs(kelvin - float(zone["native_at_reset"])) < 0.001
print("VAV reset:", round(kelvin, 4), "K")
ahu = next(row for row in fields if row["field"] == "ahs_ahu_1_supply_air_temperature_sensor")
print("AHU:", ahu["normalization"], ahu["native_unit"], ahu["observed_at_reset"])


Số dòng: 807
{('building_train_1/fairbanks_1', 'observation'): 352, ('building_train_1/fairbanks_1', 'discrete_action'): 5, ('building_train_1/fairbanks_1', 'continuous_action'): 92, ('building_train_8/fairbanks_8', 'observation'): 283, ('building_train_8/fairbanks_8', 'discrete_action'): 4, ('building_train_8/fairbanks_8', 'continuous_action'): 71}
VAV reset: 294.0 K
AHU: identity K 298.0


## 2. Chronology và leakage
Transition là `(obs_t, action_t, obs_t+1)`: quan sát hiện tại → ra lệnh → bước mô phỏng → quan sát tiếp theo. `obs_t+1` là nhãn dự báo, không phải feature của lần ra lệnh trước. `assert` sẽ dừng khi giả định sai. B01 kiểm tra hai bước; phần B03 dưới đây kiểm tra toàn bộ trajectory, bad rows và action variation. Group split theo building tránh chia các dòng gần nhau vào cả train và validation.


In [3]:
from team_tools.audit_data import verify_chronology
checks = json.loads((ROOT / "team/data_audit_checks.json").read_text(encoding="utf-8"))
for scenario in checks["scenarios"]:
    for transition in scenario["transitions"]:
        verify_chronology(transition["t"], transition["next_t"], 300.)
        assert transition["next_observation_finite"] and transition["reward_finite"]
    print(scenario["scenario"], len(scenario["transitions"]), "transition hợp lệ")


building_train_1/fairbanks_1 2 transition hợp lệ
building_train_8/fairbanks_8 2 transition hợp lệ


## 3. B02: tự tính score và thử trường hợp lỗi
Đây là **fixture tổng hợp**, không phải điểm đo từ simulator. Hai reward −8.640 và −17.280 cho loss 1 và 2; trung bình bằng 1,5. `try/except` xử lý dữ liệu bị từ chối. Lấy trung bình từng scenario để giữ trọng số ngang nhau; không trộn smoke 288 vào episode 8.640 bước.


In [4]:
import tempfile
from team_tools.benchmark import score_results, BenchmarkError
pairs = [{"building": "building_demo", "weather": name} for name in ["weather_a", "weather_b"]]
with tempfile.TemporaryDirectory() as temp:
    output = Path(temp)
    for pair, reward in zip(pairs, [-8640., -17280.]):
        file = output / pair["building"] / pair["weather"] / "pid_water/summary.json"
        file.parent.mkdir(parents=True)
        file.write_text(json.dumps({"policy": "pid_water", "total_steps": 8640, "total_reward": reward}), encoding="utf-8")
    result = score_results(output, pairs, "pid_water")
    assert result["mean_loss"] == 1.5
    print("Toy fixture mean_loss:", result["mean_loss"])
    file.unlink()  # Chỉ xóa file fixture trong thư mục tạm của chính cell.
    try:
        score_results(output, pairs, "pid_water")
    except BenchmarkError as error:
        print("Từ chối thiếu scenario:", error)
    else:
        raise AssertionError("Scorer đã cho qua một suite thiếu scenario")


Toy fixture mean_loss: 1.5
Từ chối thiếu scenario: missing scenarios: building_demo/weather_b


## 4. Bằng chứng thật và quyết định thực nghiệm
Registry lưu run ID, SHA/config, bước, runtime và trạng thái. Ô điểm trống nghĩa là chưa có score suite, không phải điểm 0. Khác phiên bản simulator thì không ghép thành kết luận cải thiện. File kiểm chứng ghi riêng lịch sử nguồn và các giới hạn đã biết.


In [5]:
with (ROOT / "experiments/registry.csv").open(encoding="utf-8", newline="") as stream:
    records = list(csv.DictReader(stream))
for row in records:
    print(row["experiment_id"], row["status"], row["step_count"], "steps; mean_loss=", row["mean_loss"] or "NOT SCORED")
report = ROOT / "team/benchmark_checks.json"
if report.exists():
    print(json.loads(report.read_text(encoding="utf-8"))["baseline_reference_audit"])


b02-smoke-20261120-01 FAILED 288 steps; mean_loss= NOT SCORED
b02-smoke-20261120-02 SUCCEEDED_SMOKE 288 steps; mean_loss= NOT SCORED
b02-full1-20261120-01 SUCCEEDED_FULL 8640 steps; mean_loss= NOT SCORED
b02-smoke-20261121-01 SUCCEEDED_SMOKE 288 steps; mean_loss= NOT SCORED
b02-reference-version-audit-20261121-01 SUCCEEDED_FULL 8640 steps; mean_loss= NOT SCORED
b02-full-repeat-20261121-01 SUCCEEDED_FULL 8640 steps; mean_loss= NOT SCORED
{'hypothesis': 'The shipped reference was measured before upstream physics fixes; same policy/config and dependencies on initial source should recover the shipped reference.', 'decision_rule': 'Compare reward to shipped reference with absolute tolerance 1e-6; otherwise retain unresolved mismatch.', 'historical_upstream_sha': '3bb6c3265d8c361e3aad518525773b3db0da8dbe', 'shipped_reference_total_reward': -1828.9035575342423, 'historical_run_total_reward': -1828.9035575342423, 'absolute_error': 0.0, 'historical_full_steps': 8640, 'runtime_wall_s': 361.25020

## Lộ trình
B01 kiểm tra semantics; B02 kiểm tra runner/scorer; B03 thu trajectory thật, ghi schema và chất lượng dữ liệu. B04 mới đánh giá persistence và Ridge, group split theo building. MAE thấp chưa chứng minh policy điều khiển tốt: cần cùng Base đã khóa, full scenarios và paired loss. Holdout chỉ mở khi nhóm quyết định audit cuối; notebook này không đọc holdout.


## 5. B03: corpus transition thật và split theo building
Một episode NPZ giữ T+1 state, T action và T nhãn. Chọn feature tại t, target tại t+1. Mỗi building có D/C/K/Z riêng; bước B04 sẽ chuẩn hóa schema feature per-zone, còn ID chỉ để group/split/diagnostics. Cell dưới chỉ đọc dataset đã hoàn tất và validate checksum; không chạy simulator hoặc train model.


In [6]:
from team_tools.dataset import validate_corpus
CORPUS = ROOT / "data/generated/trajectories/b03-corpus-20261123-02"
if CORPUS.exists() and json.loads((CORPUS / "dataset.json").read_text())["status"] == "SUCCEEDED_FULL":
    b03 = validate_corpus(CORPUS)
    print("Transitions:", b03["total_transitions"], "; zone samples:", b03["total_zone_transition_samples"])
    print("Group split:", b03["splits"])
    for episode in b03["episodes"]:
        print(episode["scenario"], episode["split"], "D=", episode["observation_dim"], "Z=", episode["zones"], "missing=", episode["raw_invalid_sensor_values"])
else:
    b03 = None
    print("B03 corpus NOT AVAILABLE locally yet; reproduce using README commands.")


Transitions: 51840 ; zone samples: 3568320
Group split: {'train': ['building_train_1', 'building_train_2', 'building_train_4'], 'validation': ['building_train_5', 'building_train_7'], 'generated_dev': ['building_0500']}
building_train_1/fairbanks_1 train D= 352 Z= 39 missing= 0
building_train_2/denver_8 train D= 703 Z= 85 missing= 0
building_train_4/phoenix_8 train D= 472 Z= 54 missing= 0
building_train_5/atlanta_6 validation D= 652 Z= 79 missing= 0
building_train_7/singapore_6 validation D= 334 Z= 36 missing= 0
building_0500/atlanta_4 generated_dev D= 958 Z= 120 missing= 0


## 6. Xem một transition và tách X/y
NumPy array dùng index bắt đầu từ 0; slicing `[:-1]` lấy mọi state trừ state cuối, `[1:]` lấy mọi state trừ state đầu. Đây là cách ghép thời gian đúng mà không random shuffle trước khi split. `allow_pickle=False` đọc NPZ không có object Python.


In [7]:
import numpy as np
if b03 is not None:
    first = CORPUS / "episodes/building_train_1__fairbanks_1"
    meta = json.loads((first / "metadata.json").read_text())
    with np.load(first / "arrays.npz", allow_pickle=False) as data:
        t, zone = 1, 0
        print({"time": str(data["timestamps"][t]), "next_time": str(data["timestamps"][t+1]),
               "zone": meta["zone_ids"][zone], "T_t_K": float(data["zone_temperature_K"][t,zone]),
               "T_next_K": float(data["zone_temperature_K"][t+1,zone]), "label_delta_K": float(data["delta_temperature_K"][t,zone])})
        assert np.isclose(data["delta_temperature_K"][t,zone], data["zone_temperature_K"][t+1,zone] - data["zone_temperature_K"][t,zone], atol=1e-6)
        # Ví dụ feature đơn giản để hiểu shape, chưa fit mô hình.
        X_temperature = data["zone_temperature_K"][:-1].reshape(-1,1)
        y_delta = data["delta_temperature_K"].reshape(-1)
        usable = data["target_valid"].reshape(-1)
        assert len(X_temperature) == len(y_delta) == len(usable)
        print("X/y/mask shape:", X_temperature.shape, y_delta.shape, usable.shape)


{'time': '2023-01-01T06:05:00', 'next_time': '2023-01-01T06:10:00', 'zone': 'VAV_1', 'T_t_K': 293.99999998509884, 'T_next_K': 294.06240843236446, 'label_delta_K': 0.062408447265625}
X/y/mask shape: (336960, 1) (336960,) (336960,)


## 7. Đọc action variation và giới hạn kết luận
Command có thể bị clamps/động học thiết bị làm measured position khác đi. Quan sát nhiều action khác nhau chỉ chứng minh dữ liệu có variation; chưa xác định nhân quả. Reward dùng để kiểm tra rollout, không làm feature hay prediction target. B03 không có MAE/RMSE/model artifact; B04 sẽ bắt đầu bằng persistence ΔT_hat=0 và Ridge, fit trên train, đánh giá riêng validation/generated-dev. Holdout vẫn đóng.

Pha sensor: VAV được latch trước thermal solve. Nhãn ΔT là hiệu sensor liên tiếp theo API; action_t có thể chỉ hiện tác động ở sensor sau nữa. B04 nên kiểm tra feature previous action/horizon bằng dữ liệu quá khứ, không dùng hidden physical state để thay nhãn.


In [8]:
if b03 is not None:
    for episode in b03["episodes"]:
        switches = sum(episode["discrete_switches_per_channel"])
        print(episode["scenario"], "continuous channels changed:", episode["continuous_channels_changed"], "discrete switches:", switches,
              "masked targets:", episode["masked_zone_targets"], "dropped rows:", episode["rows_dropped"])


building_train_1/fairbanks_1 continuous channels changed: 88 discrete switches: 5163 masked targets: 0 dropped rows: 0
building_train_2/denver_8 continuous channels changed: 193 discrete switches: 9290 masked targets: 0 dropped rows: 0
building_train_4/phoenix_8 continuous channels changed: 110 discrete switches: 5734 masked targets: 0 dropped rows: 0
building_train_5/atlanta_6 continuous channels changed: 176 discrete switches: 15682 masked targets: 0 dropped rows: 0
building_train_7/singapore_6 continuous channels changed: 82 discrete switches: 13543 masked targets: 0 dropped rows: 0
building_0500/atlanta_4 continuous channels changed: 163 discrete switches: 20542 masked targets: 0 dropped rows: 0


# B04: persistence và Ridge — 25/11/2026
Persistence dự đoán nhiệt độ lần sau bằng hiện tại, tức ΔT = 0. Ridge học một tổng có trọng số của các đặc trưng hiện tại/quá khứ và phạt hệ số lớn.
Chỉ train buildings 1/2/4 được fit scaler và hệ số. Buildings 5/7 là validation; generated 0500 chỉ đánh giá. Các ID chỉ dùng ánh xạ và báo cáo.
MAE là trung bình sai số tuyệt đối (K); RMSE phạt sai số lớn mạnh hơn. Dự báo tốt chưa chứng minh điều khiển tốt.
Ridge cải thiện validation nhưng MAE generated-dev xấu hơn persistence. Không bật B05, không tune theo generated-dev trong lượt này.


In [ ]:
from pathlib import Path
import json
b04_root = Path.cwd() if (Path.cwd()/"team_tools").exists() else Path.cwd().parent
b04 = json.loads((b04_root/"team/B04_checks.json").read_text(encoding="utf-8"))
for row in b04["metrics"]:
    print(row["split"], row["building"], row["model"], "MAE K:", round(row["mae_K"], 4), "RMSE K:", round(row["rmse_K"], 4))
